In [ ]:
# %% [markdown]
# # Script gộp danh sách các file JS
# Script này sẽ đọc các file chứa array object `{ text, link }`,
# loại bỏ đuôi `.md` ở link, và gộp chung lại thành một file `searchItems.js`.

# %%
import os
import re
import glob

def merge_js_files(input_files, output_file):
    """
    Hàm đọc và gộp danh sách các file .js

    Args:
        input_files (list): Danh sách đường dẫn tới các file .js đầu vào
        output_file (str): Đường dẫn file .js đầu ra
    """
    all_items = []

    # Biểu thức Regex để bắt các object có định dạng: { text: "...", link: "..." }
    # Hỗ trợ cả dấu nháy đơn (') và nháy kép (")
    pattern = re.compile(r'\{\s*text\s*:\s*(["\'])(.*?)\1\s*,\s*link\s*:\s*(["\'])(.*?)\3\s*\}')

    for file_path in input_files:
        if not os.path.exists(file_path):
            print(f"⚠️ Cảnh báo: Không tìm thấy file {file_path}")
            continue

        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

            # Tìm tất cả các khớp (matches) trong file
            matches = pattern.findall(content)

            for match in matches:
                # match[1] là giá trị của text, match[3] là giá trị của link
                text_value = match[1]
                link_value = match[3]

                # Xóa đuôi .md nếu có
                if link_value.endswith('.md'):
                    link_value = link_value[:-3]

                all_items.append({
                    "text": text_value,
                    "link": link_value
                })

        print(f"✅ Đã đọc file: {file_path} (Tìm thấy {len(matches)} items)")

    # Tiến hành ghi ra file gộp (output)
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('export const searchItems = [\n')

        for i, item in enumerate(all_items):
            # Escape các ký tự nháy kép (") trong text để tránh gây lỗi cú pháp JS
            safe_text = item["text"].replace('"', '\\"')
            safe_link = item["link"]

            # Tạo chuỗi format cho từng dòng
            line = f'    {{ text: "{safe_text}", link: "{safe_link}" }}'

            # Thêm dấu phẩy ở cuối nếu chưa phải item cuối cùng
            if i < len(all_items) - 1:
                line += ','

            f.write(line + '\n')

        f.write('];\n')

    print(f"\n🎉 HOÀN TẤT! Đã gộp thành công {len(all_items)} items vào file: '{output_file}'")

# %%
# ==========================================
# CẤU HÌNH VÀ CHẠY THỬ NGHIỆM
# ==========================================

# CÁCH 1: Liệt kê thủ công từng file
input_files = [
  #  '../docs/kinhtruongbo/thichminhchau/meta/filelist.js',
  #  '../docs/kinhtruongbo/sujato-vi/meta/filelist.js',
  #  '../docs/kinhtruongbo/pali-vi/meta/filelist.js',
    '../docs/kinhtruongbo/c-pali-tmc-vi/filelist.js',
    '../docs/kinhtruongbo/c-sujato-tmc-vi/filelist.js',

    # '../docs/kinhtrungbo/thichminhchau/meta/filelist.js',
    # '../docs/kinhtrungbo/sujato-vi/meta/filelist.js',
    # '../docs/kinhtrungbo/pali-vi/meta/filelist.js',
    '../docs/kinhtrungbo/c-pali-tmc-vi/filelist.js',
    '../docs/kinhtrungbo/c-sujato-tmc-vi/filelist.js',

    # '../docs/kinhtangchi/thichminhchau/meta/filelist.js',
    '../docs/kinhtangchi/sujato-vi/filelist.js',
    '../docs/kinhtangchi/c-sujato-tmc-vi/filelist.js',

    # '../docs/kinhtuongung/thichminhchau/meta/filelist.js',
    '../docs/kinhtuongung/sujato-vi/filelist.js',
    '../docs/kinhtuongung/c-sujato-tmc-vi/filelist.js',

    # Thêm đường dẫn các file của bạn vào đây...
]

# CÁCH 2: Quét toàn bộ file .js trong một thư mục (Bỏ comment dòng dưới để dùng)
# input_files = glob.glob('./thu_muc_chua_js/*.js')

# Tên file xuất ra
output_file = '../docs/search-items.js'

# Gọi hàm
merge_js_files(input_files, output_file)

In [4]:
import os
import re
import json

def extract_base_path(file_path):
    """
    Trích base path từ đường dẫn file input.
    Ví dụ: '../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js'
            → '/kinhtruongbo/c-pali-tmc-vi/'
    """
    # Lấy thư mục chứa file (bỏ tên file)
    dir_path = os.path.dirname(file_path)  # '../docs/kinhtruongbo/c-pali-tmc-vi'

    # Chuẩn hóa dấu gạch chéo
    dir_path = dir_path.replace("\\", "/")

    # Bỏ prefix '../docs' hoặc './docs' hoặc 'docs' ở đầu
    dir_path = re.sub(r'^\.{0,2}/docs', '', dir_path)

    # Đảm bảo có dấu / ở đầu và cuối
    if not dir_path.startswith('/'):
        dir_path = '/' + dir_path
    if not dir_path.endswith('/'):
        dir_path = dir_path + '/'

    return dir_path  # '/kinhtruongbo/c-pali-tmc-vi/'


def merge_js_files(input_files, output_file):
    all_items = []

    # Pattern loại 1: { text: "...", link: "..." }
    pattern_type1 = re.compile(
        r'\{\s*text\s*:\s*(["\'])(.*?)\1\s*,\s*link\s*:\s*(["\'])(.*?)\3\s*\}'
    )

    # Pattern loại 2: const xxx = [...];
    pattern_type2_array = re.compile(
        r'=\s*(\[[\s\S]*?\]);', re.MULTILINE
    )

    for file_path in input_files:
        if not os.path.exists(file_path):
            print(f"⚠️ Không tìm thấy file: {file_path}")
            continue

        base_path = extract_base_path(file_path)

        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # --- Thử loại 2 (có params.slug) ---
        array_match = pattern_type2_array.search(content)
        found_type2 = False

        if array_match:
            try:
                raw_array = array_match.group(1)
                data = json.loads(raw_array)
                count = 0
                for entry in data:
                    params = entry.get("params", {})
                    slug = params.get("slug", "")
                    title = params.get("data", {}).get("title", "")
                    if slug and title:
                        # Gắn base_path vào slug
                        link_value = base_path + slug
                        all_items.append({"text": title, "link": link_value})
                        count += 1
                if count > 0:
                    found_type2 = True
                    print(f"✅ [Loại 2] {file_path} → base: '{base_path}' ({count} items)")
            except json.JSONDecodeError:
                pass

        # --- Nếu không phải loại 2, thử loại 1 ---
        if not found_type2:
            matches = pattern_type1.findall(content)
            count = 0
            for match in matches:
                text_value = match[1]
                link_value = match[3]
                if link_value.endswith('.md'):
                    link_value = link_value[:-3]
                all_items.append({"text": text_value, "link": link_value})
                count += 1
            print(f"✅ [Loại 1] {file_path} ({count} items)")

    # Ghi output
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('export const searchItems = [\n')
        for i, item in enumerate(all_items):
            safe_text = item["text"].replace('"', '\\"')
            safe_link = item["link"]
            line = f'    {{ text: "{safe_text}", link: "{safe_link}" }}'
            if i < len(all_items) - 1:
                line += ','
            f.write(line + '\n')
        f.write('];\n')

    print(f"\n🎉 HOÀN TẤT! Gộp {len(all_items)} items → '{output_file}'")


input_files = [
  #  '../docs/kinhtruongbo/thichminhchau/meta/filelist.js',
  #  '../docs/kinhtruongbo/sujato-vi/meta/filelist.js',
  #  '../docs/kinhtruongbo/pali-vi/meta/filelist.js',
    '../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js',
    '../docs/kinhtruongbo/c-sujato-tmc-vi/tmc.js',

    # '../docs/kinhtrungbo/thichminhchau/meta/filelist.js',
    # '../docs/kinhtrungbo/sujato-vi/meta/filelist.js',
    # '../docs/kinhtrungbo/pali-vi/meta/filelist.js',
    '../docs/kinhtrungbo/c-pali-tmc-vi/tmc.js',
    '../docs/kinhtrungbo/c-nm-tmc-vi/tmc.js',

    # '../docs/kinhtangchi/thichminhchau/meta/filelist.js',
    # '../docs/kinhtangchi/sujato-vi/filelist.js',
    '../docs/kinhtangchi/c-sujato-tmc-vi/tmc.js',

    # '../docs/kinhtuongung/thichminhchau/meta/filelist.js',
    # '../docs/kinhtuongung/sujato-vi/filelist.js',
    '../docs/kinhtuongung/c-sujato-tmc-vi/tmc.js',

    # Thêm đường dẫn các file của bạn vào đây...
]

# Tên file xuất ra
output_file = '../docs/search-items.js'

# Gọi hàm
merge_js_files(input_files, output_file)

✅ [Loại 2] ../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js → base: '/kinhtruongbo/c-pali-tmc-vi/' (34 items)
✅ [Loại 2] ../docs/kinhtruongbo/c-sujato-tmc-vi/tmc.js → base: '/kinhtruongbo/c-sujato-tmc-vi/' (34 items)
✅ [Loại 2] ../docs/kinhtrungbo/c-pali-tmc-vi/tmc.js → base: '/kinhtrungbo/c-pali-tmc-vi/' (152 items)
✅ [Loại 2] ../docs/kinhtrungbo/c-nm-tmc-vi/tmc.js → base: '/kinhtrungbo/c-nm-tmc-vi/' (152 items)
✅ [Loại 2] ../docs/kinhtangchi/c-sujato-tmc-vi/tmc.js → base: '/kinhtangchi/c-sujato-tmc-vi/' (186 items)
✅ [Loại 2] ../docs/kinhtuongung/c-sujato-tmc-vi/tmc.js → base: '/kinhtuongung/c-sujato-tmc-vi/' (56 items)

🎉 HOÀN TẤT! Gộp 614 items → '../docs/search-items.js'


In [5]:
import os
import re
import json
from datetime import datetime

# ─── CẤU HÌNH ────────────────────────────────────────────────
SITE_BASE_URL = "https://kinhnikaya.org"   # ← đổi thành domain thực của bạn
SITEMAP_CHANGEFREQ = "weekly"           # always | hourly | daily | weekly | monthly | yearly | never
SITEMAP_PRIORITY = "0.7"                 # 0.0 → 1.0
# ─────────────────────────────────────────────────────────────


def extract_base_path(file_path):
    dir_path = os.path.dirname(file_path)
    dir_path = dir_path.replace("\\", "/")
    dir_path = re.sub(r'^\.{0,2}/docs', '', dir_path)
    if not dir_path.startswith('/'):
        dir_path = '/' + dir_path
    if not dir_path.endswith('/'):
        dir_path = dir_path + '/'
    return dir_path


def parse_items_from_file(file_path):
    """Trả về list of { text, link } từ một file .js"""
    if not os.path.exists(file_path):
        print(f"⚠️  Không tìm thấy file: {file_path}")
        return []

    base_path = extract_base_path(file_path)
    items = []

    pattern_type1 = re.compile(
        r'\{\s*text\s*:\s*(["\'])(.*?)\1\s*,\s*link\s*:\s*(["\'])(.*?)\3\s*\}'
    )
    pattern_type2_array = re.compile(r'=\s*(\[[\s\S]*?\]);', re.MULTILINE)

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # --- Loại 2 (params.slug) ---
    array_match = pattern_type2_array.search(content)
    found_type2 = False

    if array_match:
        try:
            data = json.loads(array_match.group(1))
            count = 0
            for entry in data:
                params = entry.get("params", {})
                slug = params.get("slug", "")
                title = params.get("data", {}).get("title", "")
                if slug and title:
                    items.append({"text": title, "link": base_path + slug})
                    count += 1
            if count > 0:
                found_type2 = True
                print(f"✅ [Loại 2] {file_path} → base: '{base_path}' ({count} items)")
        except json.JSONDecodeError:
            pass

    # --- Loại 1 ({ text, link }) ---
    if not found_type2:
        matches = pattern_type1.findall(content)
        for match in matches:
            link_value = match[3]
            if link_value.endswith('.md'):
                link_value = link_value[:-3]
            items.append({"text": match[1], "link": link_value})
        print(f"✅ [Loại 1] {file_path} ({len(matches)} items)")

    return items


def write_search_file(all_items, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('export const searchItems = [\n')
        for i, item in enumerate(all_items):
            safe_text = item["text"].replace('"', '\\"')
            line = f'    {{ text: "{safe_text}", link: "{item["link"]}" }}'
            if i < len(all_items) - 1:
                line += ','
            f.write(line + '\n')
        f.write('];\n')
    print(f"📄 Search file  → '{output_file}' ({len(all_items)} items)")


def write_sitemap_file(all_items, output_file, base_url, changefreq, priority):
    today = datetime.utcnow().strftime("%Y-%m-%d")
    base_url = base_url.rstrip('/')

    lines = ['<?xml version="1.0" encoding="UTF-8"?>']
    lines.append('<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">')

    for item in all_items:
        link = item["link"]
        # Đảm bảo link bắt đầu bằng /
        if not link.startswith('/'):
            link = '/' + link
        full_url = base_url + link

        lines.append('  <url>')
        lines.append(f'    <loc>{full_url}</loc>')
        lines.append(f'    <lastmod>{today}</lastmod>')
        lines.append(f'    <changefreq>{changefreq}</changefreq>')
        lines.append(f'    <priority>{priority}</priority>')
        lines.append('  </url>')

    lines.append('</urlset>')

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    print(f"🗺️  Sitemap file → '{output_file}' ({len(all_items)} URLs)")


def merge_js_files(input_files, search_output, sitemap_output):
    all_items = []

    for file_path in input_files:
        all_items.extend(parse_items_from_file(file_path))

    write_search_file(all_items, search_output)
    write_sitemap_file(
        all_items,
        sitemap_output,
        base_url=SITE_BASE_URL,
        changefreq=SITEMAP_CHANGEFREQ,
        priority=SITEMAP_PRIORITY,
    )

    print(f"\n🎉 HOÀN TẤT! Tổng {len(all_items)} items → 2 files được tạo.")




input_files = [
  #  '../docs/kinhtruongbo/thichminhchau/meta/filelist.js',
  #  '../docs/kinhtruongbo/sujato-vi/meta/filelist.js',
  #  '../docs/kinhtruongbo/pali-vi/meta/filelist.js',
    '../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js',
    '../docs/kinhtruongbo/c-sujato-tmc-vi/tmc.js',

    # '../docs/kinhtrungbo/thichminhchau/meta/filelist.js',
    # '../docs/kinhtrungbo/sujato-vi/meta/filelist.js',
    # '../docs/kinhtrungbo/pali-vi/meta/filelist.js',
    '../docs/kinhtrungbo/c-pali-tmc-vi/tmc.js',
    '../docs/kinhtrungbo/c-nm-tmc-vi/tmc.js',

    # '../docs/kinhtangchi/thichminhchau/meta/filelist.js',
    # '../docs/kinhtangchi/sujato-vi/filelist.js',
    '../docs/kinhtangchi/c-sujato-tmc-vi/tmc.js',

    # '../docs/kinhtuongung/thichminhchau/meta/filelist.js',
    # '../docs/kinhtuongung/sujato-vi/filelist.js',
    '../docs/kinhtuongung/c-sujato-tmc-vi/tmc.js',

    # Thêm đường dẫn các file của bạn vào đây...
]


# Tên file xuất ra
search_file = '../docs/search-items.js'
sitemap_file = "../docs/public/sitemap.xml"

merge_js_files(
    input_files=input_files,
    search_output=search_file,
    sitemap_output=sitemap_file,
)

✅ [Loại 2] ../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js → base: '/kinhtruongbo/c-pali-tmc-vi/' (34 items)
✅ [Loại 2] ../docs/kinhtruongbo/c-sujato-tmc-vi/tmc.js → base: '/kinhtruongbo/c-sujato-tmc-vi/' (34 items)
✅ [Loại 2] ../docs/kinhtrungbo/c-pali-tmc-vi/tmc.js → base: '/kinhtrungbo/c-pali-tmc-vi/' (152 items)
✅ [Loại 2] ../docs/kinhtrungbo/c-nm-tmc-vi/tmc.js → base: '/kinhtrungbo/c-nm-tmc-vi/' (152 items)
✅ [Loại 2] ../docs/kinhtangchi/c-sujato-tmc-vi/tmc.js → base: '/kinhtangchi/c-sujato-tmc-vi/' (186 items)
✅ [Loại 2] ../docs/kinhtuongung/c-sujato-tmc-vi/tmc.js → base: '/kinhtuongung/c-sujato-tmc-vi/' (56 items)
📄 Search file  → '../docs/search-items.js' (614 items)
🗺️  Sitemap file → '../docs/public/sitemap.xml' (614 URLs)

🎉 HOÀN TẤT! Tổng 614 items → 2 files được tạo.


/var/folders/0r/x3qsqbx96053svmhzm1mvb_00000gp/T/ipykernel_95911/804220605.py:89: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().strftime("%Y-%m-%d")
